In [ ]:
import os
import json
import yaml
import mmdet
from mmengine.config import Config
from mmengine.runner import Runner
from mmdet.utils import register_all_modules
from mmdet.datasets import CocoDataset

def ensure_dir_exists(path):
    """디렉토리가 존재하는지 확인하고 없으면 생성"""
    if not os.path.exists(path):
        os.makedirs(path)

def load_dataset_config(config_path):
    """YAML에서 데이터셋 설정만 로드"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"YAML config file not found at {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

def validate_dataset_paths(dataset_cfg):
    """데이터셋 경로 유효성 검사"""
    data_root = dataset_cfg['dataset']['data_root']
    print(f"\nValidating datasets with data_root: {data_root}")
    
    for split in ['train', 'valid', 'test']:
        print(f"\nValidating {split} dataset:")
        
        # 이미지 디렉토리 확인
        img_path = dataset_cfg['dataset'][split]['img_path']
        print(f"Checking image directory: {img_path}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"{split} image directory not found: {img_path}")
        
        # 이미지 파일 확인
        img_files = [f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Found {len(img_files)} image files")
        if not img_files:
            raise ValueError(f"No image files found in {img_path}")
            
        # 어노테이션 파일 확인
        ann_file = dataset_cfg['dataset'][split]['ann_file']
        print(f"Checking annotation file: {ann_file}")
        if not os.path.exists(ann_file):
            raise FileNotFoundError(f"{split} annotation file not found: {ann_file}")
            
        # 어노테이션 파일 내용 확인
        with open(ann_file, 'r', encoding='utf-8') as f:
            try:
                annotations = json.load(f)
                print(f"Annotation file structure:")
                print(f"- Number of images: {len(annotations['images'])}")
                print(f"- Number of annotations: {len(annotations['annotations'])}")
                print(f"- Number of categories: {len(annotations['categories'])}")
                
                # 이미지 ID와 파일 존재 여부 확인
                for img in annotations['images']:
                    img_file = img['file_name']
                    full_path = os.path.join(img_path, img_file)
                    if not os.path.exists(full_path):
                        raise FileNotFoundError(f"Image file not found: {full_path}")
                
                if len(annotations['images']) == 0 or len(annotations['annotations']) == 0:
                    raise ValueError(f"Empty {split} dataset detected")
                    
                # 카테고리 확인
                categories = {cat['id']: cat['name'] for cat in annotations['categories']}
                print(f"- Categories: {categories}")
                
                # 클래스 일치 확인
                config_classes = set(dataset_cfg['dataset']['classes'])
                anno_classes = set(cat['name'] for cat in annotations['categories'])
                if config_classes != anno_classes:
                    print(f"Warning: Config classes ({config_classes}) do not match annotation classes ({anno_classes})")
                
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format in {ann_file}: {str(e)}")
            except KeyError as e:
                raise ValueError(f"Missing required key in annotation file: {str(e)}")
                
        print(f"{split} dataset validation completed successfully")

def create_dataset_config(dataset_cfg, split='train', pipeline=None):
    """데이터셋 설정 생성"""
    # 절대 경로에서 상대 경로로 변환
    data_root = dataset_cfg['dataset']['data_root']
    img_path = dataset_cfg['dataset'][split]['img_path']
    ann_file = dataset_cfg['dataset'][split]['ann_file']
    
    # 상대 경로 계산
    rel_img_path = os.path.relpath(img_path, data_root)
    rel_ann_file = os.path.relpath(ann_file, data_root)
    
    print(f"\nDataset config for {split}:")
    print(f"data_root: {data_root}")
    print(f"rel_img_path: {rel_img_path}")
    print(f"rel_ann_file: {rel_ann_file}")
    
    # 클래스 이름을 소문자로 통일
    classes = [cls.lower() for cls in dataset_cfg['dataset']['classes']]
    print(f"classes: {classes}")
    
    dataset_dict = dict(
        type='CocoDataset',  # 필수 type 필드 추가
        data_root=data_root,
        ann_file=rel_ann_file,
        data_prefix=dict(img=rel_img_path),
        test_mode=(split != 'train'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32) if split == 'train' else None,
        metainfo=dict(classes=classes),
        pipeline=pipeline
    )
    
    return dataset_dict

def setup_model_config(cfg_data):
    """모델 및 학습 설정"""
    # Pipeline 설정 추가
    train_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(type='RandomFlip', prob=0.5),
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='PackDetInputs')
    ]

    test_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(
            type='PackDetInputs',
            meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor'))
    ]
    # MMDetection 기본 config 사용
    base_config = os.path.join(os.path.dirname(mmdet.__file__), '..', 'configs', 'mask_rcnn', 'mask-rcnn_r50_fpn_1x_coco.py')
    
    print(f"Base config path: {base_config}")
    
    if not os.path.exists(base_config):
        print(f"Primary config not found at {base_config}, trying alternative path...")
        base_config = 'configs/mask_rcnn/mask-rcnn_r50_fpn_1x_coco.py'
        if not os.path.exists(base_config):
            raise FileNotFoundError("Model config file not found")
    
    print(f"Loading config from: {base_config}")
    cfg = Config.fromfile(base_config)
    
    # 모델 가중치 경로 설정
    cfg.load_from = 'https://download.openmmlab.com/mmdetection/v3.0/convnext/mask-rcnn_convnext-t_fpn_3x_coco/mask_rcnn_convnext-t_fpn_3x_coco_20211206_072928-20876e97.pth'
    
    # 데이터셋 설정
    cfg.train_dataloader = dict(
        batch_size=cfg_data['training']['batch_size'],
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        batch_sampler=None,  # AspectRatioBatchSampler 제거
        dataset=create_dataset_config(cfg_data, 'train', pipeline=train_pipeline)
    )
    
    cfg.val_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'valid', pipeline=test_pipeline)
    )
    
    cfg.test_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'test', pipeline=test_pipeline)
    )

    # 검증 설정
    cfg.val_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['valid']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False)

    cfg.test_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['test']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False)
    
    # 학습 설정
    cfg.train_cfg = dict(
        type='EpochBasedTrainLoop',
        max_epochs=cfg_data['training']['max_epochs'],
        val_interval=1)
    
    # 옵티마이저 설정
    cfg.optim_wrapper = dict(
        type='AmpOptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=cfg_data['training']['lr'],
            betas=(0.9, 0.999),
            weight_decay=0.05),
        clip_grad=None)
    
    # 학습률 스케줄러
    cfg.param_scheduler = [
        dict(
            type='LinearLR',
            start_factor=0.001,
            by_epoch=False,
            begin=0,
            end=500),
        dict(
            type='MultiStepLR',
            begin=0,
            end=cfg_data['training']['max_epochs'],
            by_epoch=True,
            milestones=[27, 33],
            gamma=0.1)
    ]
    
    # 체크포인트 설정
    cfg.default_hooks = dict(
        timer=dict(type='IterTimerHook'),
        logger=dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint=dict(type='CheckpointHook', 
                       interval=1,
                       max_keep_ckpts=3,
                       save_best='auto'),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='DetVisualizationHook')
    )
    
    # 작업 디렉토리 설정
    cfg.work_dir = cfg_data['model']['work_dir']
    ensure_dir_exists(cfg.work_dir)
    
    # 클래스 수 설정
    cfg.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.bbox_head.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.mask_head.num_classes = len(cfg_data['dataset']['classes'])
    
    return cfg

def train_model(yaml_config_path):
    """모델 학습 실행"""
    try:
        # .mim 디렉토리 생성
        mim_dir = os.path.join(os.path.dirname(mmdet.__file__), '.mim')
        ensure_dir_exists(mim_dir)
        
        # model-index.yml 파일 생성
        model_index_path = os.path.join(mim_dir, 'model-index.yml')
        if not os.path.exists(model_index_path):
            model_index_content = {
                'Import': [],
                'Collections': []
            }
            with open(model_index_path, 'w') as f:
                yaml.dump(model_index_content, f)
        
        # 모든 모듈 등록
        register_all_modules()
        
        # 설정 로드
        cfg_data = load_dataset_config(yaml_config_path)
        
        # 데이터셋 경로 검증
        validate_dataset_paths(cfg_data)
        
        # 모델 설정
        cfg = setup_model_config(cfg_data)
        
        # Runner 생성
        runner = Runner.from_cfg(cfg)
        
        try:
            # 학습 시작
            runner.train()
        except Exception as e:
            if 'HTTP Error 404' in str(e):
                print("체크포인트 다운로드 실패. ResNet-50 기본 가중치를 사용합니다.")
                cfg.load_from = None  # 기본 ResNet-50 가중치 사용
                runner = Runner.from_cfg(cfg)
                runner.train()
            else:
                raise
        
        # 테스트 실행
        runner.test()
        
    except Exception as e:
        print(f"Error during training: {str(e)}")
        raise

if __name__ == '__main__':
    yaml_config_path = 'C:/Users/AI-LHJ/Desktop/json dataset/dataset.yaml'  # YAML 파일 경로
    train_model(yaml_config_path)

In [ ]:
import os
import json
import yaml
import mmdet
import pandas as pd
from tabulate import tabulate
from mmengine.config import Config
from mmengine.runner import Runner
from mmdet.utils import register_all_modules
from mmdet.datasets import CocoDataset

def ensure_dir_exists(path):
    """디렉토리가 존재하는지 확인하고 없으면 생성"""
    if not os.path.exists(path):
        os.makedirs(path)

def load_dataset_config(config_path):
    """YAML에서 데이터셋 설정만 로드"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"YAML config file not found at {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

def validate_dataset_paths(dataset_cfg):
    """데이터셋 경로 유효성 검사"""
    data_root = dataset_cfg['dataset']['data_root']
    print(f"\nValidating datasets with data_root: {data_root}")
    
    for split in ['train', 'valid', 'test']:
        print(f"\nValidating {split} dataset:")
        
        # 이미지 디렉토리 확인
        img_path = dataset_cfg['dataset'][split]['img_path']
        print(f"Checking image directory: {img_path}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"{split} image directory not found: {img_path}")
        
        # 이미지 파일 확인
        img_files = [f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Found {len(img_files)} image files")
        if not img_files:
            raise ValueError(f"No image files found in {img_path}")
            
        # 어노테이션 파일 확인
        ann_file = dataset_cfg['dataset'][split]['ann_file']
        print(f"Checking annotation file: {ann_file}")
        if not os.path.exists(ann_file):
            raise FileNotFoundError(f"{split} annotation file not found: {ann_file}")
            
        # 어노테이션 파일 내용 확인
        with open(ann_file, 'r', encoding='utf-8') as f:
            try:
                annotations = json.load(f)
                print(f"Annotation file structure:")
                print(f"- Number of images: {len(annotations['images'])}")
                print(f"- Number of annotations: {len(annotations['annotations'])}")
                print(f"- Number of categories: {len(annotations['categories'])}")
                
                # 이미지 ID와 파일 존재 여부 확인
                for img in annotations['images']:
                    img_file = img['file_name']
                    full_path = os.path.join(img_path, img_file)
                    if not os.path.exists(full_path):
                        raise FileNotFoundError(f"Image file not found: {full_path}")
                
                if len(annotations['images']) == 0 or len(annotations['annotations']) == 0:
                    raise ValueError(f"Empty {split} dataset detected")
                    
                # 카테고리 확인
                categories = {cat['id']: cat['name'] for cat in annotations['categories']}
                print(f"- Categories: {categories}")
                
                # 클래스 일치 확인
                config_classes = set(dataset_cfg['dataset']['classes'])
                anno_classes = set(cat['name'] for cat in annotations['categories'])
                if config_classes != anno_classes:
                    print(f"Warning: Config classes ({config_classes}) do not match annotation classes ({anno_classes})")
                
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format in {ann_file}: {str(e)}")
            except KeyError as e:
                raise ValueError(f"Missing required key in annotation file: {str(e)}")
                
        print(f"{split} dataset validation completed successfully")

def create_dataset_config(dataset_cfg, split='train', pipeline=None):
    """데이터셋 설정 생성"""
    # 절대 경로에서 상대 경로로 변환
    data_root = dataset_cfg['dataset']['data_root']
    img_path = dataset_cfg['dataset'][split]['img_path']
    ann_file = dataset_cfg['dataset'][split]['ann_file']
    
    # 상대 경로 계산
    rel_img_path = os.path.relpath(img_path, data_root)
    rel_ann_file = os.path.relpath(ann_file, data_root)
    
    print(f"\nDataset config for {split}:")
    print(f"data_root: {data_root}")
    print(f"rel_img_path: {rel_img_path}")
    print(f"rel_ann_file: {rel_ann_file}")
    
    # 클래스 이름을 소문자로 통일
    classes = [cls.lower() for cls in dataset_cfg['dataset']['classes']]
    print(f"classes: {classes}")
    
    dataset_dict = dict(
        type='CocoDataset',  # 필수 type 필드 추가
        data_root=data_root,
        ann_file=rel_ann_file,
        data_prefix=dict(img=rel_img_path),
        test_mode=(split != 'train'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32) if split == 'train' else None,
        metainfo=dict(classes=classes),
        pipeline=pipeline
    )
    
    return dataset_dict

def setup_model_config(cfg_data):
    """모델 및 학습 설정"""
    # Pipeline 설정 추가
    train_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(type='RandomFlip', prob=0.5),
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='PackDetInputs')
    ]

    test_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(
            type='PackDetInputs',
            meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor'))
    ]
    # MMDetection 기본 config 사용
    base_config = os.path.join(os.path.dirname(mmdet.__file__), '..', 'configs', 'mask_rcnn', 'mask-rcnn_r50_fpn_1x_coco.py')
    
    print(f"Base config path: {base_config}")
    
    if not os.path.exists(base_config):
        print(f"Primary config not found at {base_config}, trying alternative path...")
        base_config = 'configs/mask_rcnn/mask-rcnn_r50_fpn_1x_coco.py'
        if not os.path.exists(base_config):
            raise FileNotFoundError("Model config file not found")
    
    print(f"Loading config from: {base_config}")
    cfg = Config.fromfile(base_config)
    
    # 모델 가중치 경로 설정
    cfg.load_from = 'https://download.openmmlab.com/mmdetection/v3.0/convnext/mask-rcnn_convnext-t_fpn_3x_coco/mask_rcnn_convnext-t_fpn_3x_coco_20211206_072928-20876e97.pth'
    
    # 데이터셋 설정
    cfg.train_dataloader = dict(
        batch_size=cfg_data['training']['batch_size'],
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        batch_sampler=None,  # AspectRatioBatchSampler 제거
        dataset=create_dataset_config(cfg_data, 'train', pipeline=train_pipeline)
    )
    
    cfg.val_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'valid', pipeline=test_pipeline)
    )
    
    cfg.test_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'test', pipeline=test_pipeline)
    )

    # 검증 설정
    cfg.val_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['valid']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False)

    # 검증 설정
    cfg.test_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['test']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False,
        classwise=True,  # 클래스별 성능 평가
        per_class_AR=True  # 클래스별 Average Recall 계산
    )
    
    # 학습 설정
    cfg.train_cfg = dict(
        type='EpochBasedTrainLoop',
        max_epochs=cfg_data['training']['max_epochs'],
        val_interval=1)
    
    # 옵티마이저 설정
    cfg.optim_wrapper = dict(
        type='AmpOptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=cfg_data['training']['lr'],
            betas=(0.9, 0.999),
            weight_decay=0.05),
        clip_grad=None)
    
    # 학습률 스케줄러
    cfg.param_scheduler = [
        dict(
            type='LinearLR',
            start_factor=0.001,
            by_epoch=False,
            begin=0,
            end=500),
        dict(
            type='MultiStepLR',
            begin=0,
            end=cfg_data['training']['max_epochs'],
            by_epoch=True,
            milestones=[27, 33],
            gamma=0.1)
    ]
    
    # 체크포인트 설정
    cfg.default_hooks = dict(
        timer=dict(type='IterTimerHook'),
        logger=dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint=dict(type='CheckpointHook', 
                       interval=1,
                       max_keep_ckpts=3,
                       save_best='auto'),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='DetVisualizationHook')
    )
    
    # 작업 디렉토리 설정
    cfg.work_dir = cfg_data['model']['work_dir']
    ensure_dir_exists(cfg.work_dir)
    
    # 클래스 수 설정
    cfg.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.bbox_head.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.mask_head.num_classes = len(cfg_data['dataset']['classes'])
    
    return cfg

def evaluate_model_performance(results, dataset_cfg):
    """모델 성능 평가 및 결과 표시"""
    # 클래스별 성능 지표 계산
    classes = dataset_cfg['dataset']['classes']
    metrics = {'Class': [], 'Precision': [], 'Recall': [], 'mAP50': [], 'mAP50-95': []}
    
    # 전체 성능 계산
    metrics['Class'].append('All')
    metrics['Precision'].append(f"{results['bbox_mAP_50'] * 100:.2f}")
    metrics['Recall'].append(f"{results['bbox_mAR_100'] * 100:.2f}")
    metrics['mAP50'].append(f"{results['bbox_mAP_50'] * 100:.2f}")
    metrics['mAP50-95'].append(f"{results['bbox_mAP'] * 100:.2f}")
    
    # 클래스별 성능 계산
    for i, class_name in enumerate(classes):
        metrics['Class'].append(class_name)
        metrics['Precision'].append(f"{results[f'bbox_precision_50_{i}'] * 100:.2f}")
        metrics['Recall'].append(f"{results[f'bbox_recall_100_{i}'] * 100:.2f}")
        metrics['mAP50'].append(f"{results[f'bbox_mAP_50_class_{i}'] * 100:.2f}")
        metrics['mAP50-95'].append(f"{results[f'bbox_mAP_class_{i}'] * 100:.2f}")
    
    # DataFrame 생성 및 표 형식으로 출력
    df = pd.DataFrame(metrics)
    print("\nModel Performance Evaluation:")
    print(tabulate(df, headers='keys', tablefmt='pipe', showindex=False))
    
    # 결과를 CSV 파일로 저장
    results_dir = os.path.join(dataset_cfg['model']['work_dir'], 'evaluation_results')
    ensure_dir_exists(results_dir)
    csv_path = os.path.join(results_dir, 'performance_metrics.csv')
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved to: {csv_path}")


def train_model(yaml_config_path):
    """모델 학습 실행"""
    try:
        # .mim 디렉토리 생성
        mim_dir = os.path.join(os.path.dirname(mmdet.__file__), '.mim')
        ensure_dir_exists(mim_dir)
        
        # model-index.yml 파일 생성
        model_index_path = os.path.join(mim_dir, 'model-index.yml')
        if not os.path.exists(model_index_path):
            model_index_content = {
                'Import': [],
                'Collections': []
            }
            with open(model_index_path, 'w') as f:
                yaml.dump(model_index_content, f)
        
        # 모든 모듈 등록
        register_all_modules()
        
        # 설정 로드
        cfg_data = load_dataset_config(yaml_config_path)
        
        # 데이터셋 경로 검증
        validate_dataset_paths(cfg_data)
        
        # 모델 설정
        cfg = setup_model_config(cfg_data)
        
        # Runner 생성
        runner = Runner.from_cfg(cfg)
        
        try:
            runner.train()
        except Exception as e:
            if 'HTTP Error 404' in str(e):
                print("체크포인트 다운로드 실패. ResNet-50 기본 가중치를 사용합니다.")
                cfg.load_from = None
                runner = Runner.from_cfg(cfg)
                runner.train()
            else:
                raise
        
        # 테스트 실행 및 성능 평가
        results = runner.test()
        evaluate_model_performance(results[0], cfg_data)  # results[0]는 첫 번째 평가 결과
        
    except Exception as e:
        print(f"Error during training or evaluation: {str(e)}")
        raise

if __name__ == '__main__':
    yaml_config_path = 'C:/Users/AI-LHJ/Desktop/json dataset/dataset.yaml'  # YAML 파일 경로
    train_model(yaml_config_path)

In [ ]:
import os
import json
import yaml
import mmdet
import pandas as pd
from tabulate import tabulate
from mmengine.config import Config
from mmengine.runner import Runner
from mmdet.utils import register_all_modules
from mmdet.datasets import CocoDataset

def ensure_dir_exists(path):
    """디렉토리가 존재하는지 확인하고 없으면 생성"""
    if not os.path.exists(path):
        os.makedirs(path)

def load_dataset_config(config_path):
    """YAML에서 데이터셋 설정만 로드"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"YAML config file not found at {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

def validate_dataset_paths(dataset_cfg):
    """데이터셋 경로 유효성 검사"""
    data_root = dataset_cfg['dataset']['data_root']
    print(f"\nValidating datasets with data_root: {data_root}")
    
    for split in ['train', 'valid', 'test']:
        print(f"\nValidating {split} dataset:")
        
        # 이미지 디렉토리 확인
        img_path = dataset_cfg['dataset'][split]['img_path']
        print(f"Checking image directory: {img_path}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"{split} image directory not found: {img_path}")
        
        # 이미지 파일 확인
        img_files = [f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Found {len(img_files)} image files")
        if not img_files:
            raise ValueError(f"No image files found in {img_path}")
            
        # 어노테이션 파일 확인
        ann_file = dataset_cfg['dataset'][split]['ann_file']
        print(f"Checking annotation file: {ann_file}")
        if not os.path.exists(ann_file):
            raise FileNotFoundError(f"{split} annotation file not found: {ann_file}")
            
        # 어노테이션 파일 내용 확인
        with open(ann_file, 'r', encoding='utf-8') as f:
            try:
                annotations = json.load(f)
                print(f"Annotation file structure:")
                print(f"- Number of images: {len(annotations['images'])}")
                print(f"- Number of annotations: {len(annotations['annotations'])}")
                print(f"- Number of categories: {len(annotations['categories'])}")
                
                # 이미지 ID와 파일 존재 여부 확인
                for img in annotations['images']:
                    img_file = img['file_name']
                    full_path = os.path.join(img_path, img_file)
                    if not os.path.exists(full_path):
                        raise FileNotFoundError(f"Image file not found: {full_path}")
                
                if len(annotations['images']) == 0 or len(annotations['annotations']) == 0:
                    raise ValueError(f"Empty {split} dataset detected")
                    
                # 카테고리 확인
                categories = {cat['id']: cat['name'] for cat in annotations['categories']}
                print(f"- Categories: {categories}")
                
                # 클래스 일치 확인
                config_classes = set(dataset_cfg['dataset']['classes'])
                anno_classes = set(cat['name'] for cat in annotations['categories'])
                if config_classes != anno_classes:
                    print(f"Warning: Config classes ({config_classes}) do not match annotation classes ({anno_classes})")
                
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format in {ann_file}: {str(e)}")
            except KeyError as e:
                raise ValueError(f"Missing required key in annotation file: {str(e)}")
                
        print(f"{split} dataset validation completed successfully")

def create_dataset_config(dataset_cfg, split='train', pipeline=None):
    """데이터셋 설정 생성"""
    # 절대 경로에서 상대 경로로 변환
    data_root = dataset_cfg['dataset']['data_root']
    img_path = dataset_cfg['dataset'][split]['img_path']
    ann_file = dataset_cfg['dataset'][split]['ann_file']
    
    # 상대 경로 계산
    rel_img_path = os.path.relpath(img_path, data_root)
    rel_ann_file = os.path.relpath(ann_file, data_root)
    
    print(f"\nDataset config for {split}:")
    print(f"data_root: {data_root}")
    print(f"rel_img_path: {rel_img_path}")
    print(f"rel_ann_file: {rel_ann_file}")
    
    # 클래스 이름을 소문자로 통일
    classes = [cls.lower() for cls in dataset_cfg['dataset']['classes']]
    print(f"classes: {classes}")
    
    dataset_dict = dict(
        type='CocoDataset',  # 필수 type 필드 추가
        data_root=data_root,
        ann_file=rel_ann_file,
        data_prefix=dict(img=rel_img_path),
        test_mode=(split != 'train'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32) if split == 'train' else None,
        metainfo=dict(classes=classes),
        pipeline=pipeline
    )
    
    return dataset_dict

def setup_model_config(cfg_data):
    """모델 및 학습 설정"""
    # Pipeline 설정 추가
    train_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(type='RandomFlip', prob=0.5),
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='PackDetInputs')
    ]

    test_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='Resize', scale=(640, 640), keep_ratio=False),  # 고정 크기로 변경
        dict(
            type='Normalize',
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375],
            to_rgb=True),
        dict(type='Pad', size=(640, 640)),  # 패딩 추가
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(
            type='PackDetInputs',
            meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor'))
    ]
    # MMDetection 기본 config 사용
    base_config = os.path.join(os.path.dirname(mmdet.__file__), '..', 'configs', 'mask_rcnn', 'mask-rcnn_r50_fpn_1x_coco.py')
    
    print(f"Base config path: {base_config}")
    
    if not os.path.exists(base_config):
        print(f"Primary config not found at {base_config}, trying alternative path...")
        base_config = 'configs/mask_rcnn/mask-rcnn_r50_fpn_1x_coco.py'
        if not os.path.exists(base_config):
            raise FileNotFoundError("Model config file not found")
    
    print(f"Loading config from: {base_config}")
    cfg = Config.fromfile(base_config)
    
    # 모델 가중치 경로 설정
    cfg.load_from = 'C:/Users/AI-LHJ/Desktop/mmdetection-main/projects/ConvNeXt-V2/configs/mask-rcnn_convnext-v2-b_fpn_lsj-3x-fcmae_coco_20230113_110947-757ee2dd.pth'
    
    # 데이터셋 설정
    cfg.train_dataloader = dict(
        batch_size=cfg_data['training']['batch_size'],
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        batch_sampler=None,  # AspectRatioBatchSampler 제거
        dataset=create_dataset_config(cfg_data, 'train', pipeline=train_pipeline)
    )
    
    cfg.val_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'valid', pipeline=test_pipeline)
    )
    
    cfg.test_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'test', pipeline=test_pipeline)
    )

    # 검증 설정
    cfg.val_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['valid']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False)

    cfg.test_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['test']['ann_file'],
        metric=['bbox', 'segm'],
        format_only=False)
    
    # 학습 설정
    cfg.train_cfg = dict(
        type='EpochBasedTrainLoop',
        max_epochs=cfg_data['training']['max_epochs'],
        val_interval=1)
    
    # 옵티마이저 설정
    cfg.optim_wrapper = dict(
        type='AmpOptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=cfg_data['training']['lr'],
            betas=(0.9, 0.999),
            weight_decay=0.05),
        clip_grad=None)
    
    # 학습률 스케줄러
    cfg.param_scheduler = [
        dict(
            type='LinearLR',
            start_factor=0.001,
            by_epoch=False,
            begin=0,
            end=500),
        dict(
            type='MultiStepLR',
            begin=0,
            end=cfg_data['training']['max_epochs'],
            by_epoch=True,
            milestones=[27, 33],
            gamma=0.1)
    ]
    
    # 체크포인트 설정
    cfg.default_hooks = dict(
        timer=dict(type='IterTimerHook'),
        logger=dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint=dict(type='CheckpointHook', 
                       interval=1,
                       max_keep_ckpts=3,
                       save_best='auto'),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='DetVisualizationHook')
    )
    
    # 작업 디렉토리 설정
    cfg.work_dir = cfg_data['model']['work_dir']
    ensure_dir_exists(cfg.work_dir)
    
    # 클래스 수 설정
    cfg.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.bbox_head.num_classes = len(cfg_data['dataset']['classes'])
    cfg.model.roi_head.mask_head.num_classes = len(cfg_data['dataset']['classes'])
    
    return cfg

def analyze_results(results_file, gt_file, show_details=True):
    """COCO format results file을 분석하고 클래스별 성능을 출력"""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import numpy as np

    # ground truth COCO 객체 생성
    coco_gt = COCO(gt_file)
    
    # 결과 로드
    coco_dt = coco_gt.loadRes(results_file)
    
    print("\n[클래스별 성능 평가]")
    print("-" * 50)
    print(f"{'Class':<15} {'AP50':<10} {'AP75':<10} {'mAP':<10}")
    print("-" * 50)
    
    # 전체 성능 평가
    for eval_type in ['bbox', 'segm']:
        cocoEval = COCOeval(coco_gt, coco_dt, eval_type)
        cocoEval.evaluate()
        cocoEval.accumulate()
        
        # 전체 성능
        stats = cocoEval.stats
        print(f"\n{eval_type.upper()} Results:")
        total_ap50 = stats[1] * 100  # AP at IoU=0.50
        total_ap75 = stats[2] * 100  # AP at IoU=0.75
        total_map = stats[0] * 100   # AP at IoU=0.50:0.95
        
        print(f"{'All':<15} {total_ap50:>8.1f}%  {total_ap75:>8.1f}%  {total_map:>8.1f}%")
        
        # 클래스별 성능
        cats = coco_gt.loadCats(coco_gt.getCatIds())
        for idx, cat in enumerate(cats):
            # 클래스별 precision matrix 추출
            precision = cocoEval.eval['precision']
            
            # AP@50
            ap50 = np.mean(precision[0, :, idx, 0, -1]) * 100
            
            # AP@75
            ap75 = np.mean(precision[5, :, idx, 0, -1]) * 100
            
            # mAP (AP@[0.5:0.95])
            map_class = np.mean(precision[:, :, idx, 0, -1]) * 100
            
            print(f"{cat['name']:<15} {ap50:>8.1f}%  {ap75:>8.1f}%  {map_class:>8.1f}%")

def train_model(yaml_config_path):
    """모델 학습 실행"""
    try:
        # .mim 디렉토리 생성
        mim_dir = os.path.join(os.path.dirname(mmdet.__file__), '.mim')
        ensure_dir_exists(mim_dir)
        
        # model-index.yml 파일 생성
        model_index_path = os.path.join(mim_dir, 'model-index.yml')
        if not os.path.exists(model_index_path):
            model_index_content = {
                'Import': [],
                'Collections': []
            }
            with open(model_index_path, 'w') as f:
                yaml.dump(model_index_content, f)
        
        # 모든 모듈 등록
        register_all_modules()
        
        # 설정 로드
        cfg_data = load_dataset_config(yaml_config_path)
        
        # 데이터셋 경로 검증
        validate_dataset_paths(cfg_data)
        
        # 모델 설정
        cfg = setup_model_config(cfg_data)
        
        # Runner 생성
        runner = Runner.from_cfg(cfg)
        
        try:
            runner.train()
        except Exception as e:
            if 'HTTP Error 404' in str(e):
                print("체크포인트 다운로드 실패. ResNet-50 기본 가중치를 사용합니다.")
                cfg.load_from = None
                runner = Runner.from_cfg(cfg)
                runner.train()
            else:
                raise
        
        # 테스트 실행 및 성능 평가
        results = runner.test()
        evaluate_model_performance(results[0], cfg_data)  # results[0]는 첫 번째 평가 결과
        
    except Exception as e:
        print(f"Error during training or evaluation: {str(e)}")
        raise

if __name__ == '__main__':
    yaml_config_path = 'C:/Users/AI-LHJ/Desktop/json dataset/dataset.yaml'  # YAML 파일 경로
    train_model(yaml_config_path)